In [ ]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from TICC_solver import TICC
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import f1_score

%matplotlib inline

In [ ]:
import random

np.random.seed(42)
random.seed(42)

# 前処理

In [ ]:
def _compute_window_band_powers(vals, fs, window_size, bands, center=True, pad=True, window_type='hann'):
    n = len(vals)
    if n == 0:
        return np.zeros((0, len(bands)))

    # 窓関数
    if window_type in ['hann', 'hanning']:
        win = np.hanning(window_size)
    else:
        win = np.ones(window_size)

    # 周波数軸（必ず window_size を使う）
    freqs = np.fft.rfftfreq(window_size, d=1.0/fs)

    band_mat = np.zeros((n, len(bands)), dtype=float)

    for i in range(n):
        # ウィンドウ位置決定
        if center:
            start = i - window_size // 2
        else:
            start = i
        end = start + window_size

        # 切り出し＋パディング
        if start < 0 or end > n:
            if pad:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                left = s_idx - start
                right = end - e_idx
                seg_padded = np.pad(seg, (max(0, left), max(0, right)), mode='reflect')

                if len(seg_padded) < window_size:
                    seg_padded = np.pad(seg_padded, (0, window_size - len(seg_padded)), mode='edge')

                windowed = seg_padded[:window_size]
            else:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                if len(seg) < window_size:
                    windowed = np.pad(seg, (0, window_size - len(seg)), mode='constant')
                else:
                    windowed = seg[:window_size]
        else:
            windowed = vals[start:end]

        # DC成分除去
        windowed = windowed - np.mean(windowed)

        # 窓関数適用
        windowed = windowed * win

        # --- FFT ---
        fft_vals = np.fft.rfft(windowed)

        # 振幅（正規化：2/N）
        amp = (2.0 / window_size) * np.abs(fft_vals)

        # パワー
        power = amp ** 2

        # --- バンドごとに合計 ---
        for b_i, (f_lo, f_hi) in enumerate(bands):
            mask = (freqs >= f_lo) & (freqs < f_hi)
            band_mat[i, b_i] = float(np.sum(power[mask])) if np.any(mask) else 0.0

    return band_mat


# 加速度前処理（特徴量作成）
def preprocess_acc_sensor_minimal(df_raw, sensor_suffix="acc1",
                                  fs=100.0, window_size=1024,
                                  bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
                                  center=True, pad=True,
                                  window_stat=100):
    df = df_raw.copy()

    # ---- 時刻処理 ----
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce")
    df["ArriveTime"] = df["ArriveTime"].ffill()
    df["time"] = df["ArriveTime"]

    # ---- 加速度を float へ ----
    ax = pd.to_numeric(df.get("AccelerationX", 0), errors="coerce").fillna(0.0).astype(float)
    ay = pd.to_numeric(df.get("AccelerationY", 0), errors="coerce").fillna(0.0).astype(float)
    az = pd.to_numeric(df.get("AccelerationZ", 0), errors="coerce").fillna(0.0).astype(float)

    out = pd.DataFrame(index=df.index)

    # ---- 基本値 ----
    out[f"time_{sensor_suffix}"] = df["time"].values
    out[f"acc_x_{sensor_suffix}"] = ax
    out[f"acc_y_{sensor_suffix}"] = ay
    out[f"acc_z_{sensor_suffix}"] = az

    # ---- magnitude ----
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    out[f"acc_mag_{sensor_suffix}"] = mag

    # ---- 差分 ----
    out[f"acc_mag_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(mag)]
    out[f"acc_x_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ax)]
    out[f"acc_y_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ay)]
    out[f"acc_z_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(az)]

    # ============================================
    #   ★★★ 追加した特徴量 ★★★
    # ============================================

    # --- mean_x / mean_y / mean_z（移動平均） ---
    out[f"mean_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).mean()
    out[f"mean_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).mean()
    out[f"mean_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).mean()

    # --- std_x / std_y / std_z（移動 std） ---
    out[f"std_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- mean_mag / std_mag ---
    out[f"mean_mag_{sensor_suffix}"] = mag.rolling(window_stat, min_periods=1).mean()
    out[f"std_mag_{sensor_suffix}"]  = mag.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- SMA（瞬間値）---
    out[f"sma_{sensor_suffix}"] = (np.abs(ax) + np.abs(ay) + np.abs(az)) / 3.0

    # --- int_abs（mag の積分値）---
    # dt = 1.0 / fs
    # out[f"int_abs_{sensor_suffix}"] = np.cumsum(np.abs(mag)) * dt

    # ============================================
    #   ここまで追加特徴量
    # ============================================

    # ---- バンドパワー ----
    band_x = _compute_window_band_powers(ax.values, fs, window_size, bands, center, pad)
    band_y = _compute_window_band_powers(ay.values, fs, window_size, bands, center, pad)
    band_z = _compute_window_band_powers(az.values, fs, window_size, bands, center, pad)
    band_m = _compute_window_band_powers(mag, fs, window_size, bands, center, pad)

    for b_i, (f_lo, f_hi) in enumerate(bands):
        bi = b_i + 1
        out[f"acc_x_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_x[:, b_i]
        out[f"acc_y_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_y[:, b_i]
        out[f"acc_z_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_z[:, b_i]
        out[f"acc_mag_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_m[:, b_i]

    return out


In [ ]:
def load_acc_only(
        keyword,
        root_folder=r"../../../../../デスクトップ/実験",
        fs=100.0,
        window_size=1024,
        bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
        center=True,
        pad=True,
        interpolate_missing=True,
        save_path=None):

    keyword = str(keyword)

    # 加速度フォルダ
    acc_path = os.path.join(root_folder, "data_acc")

    # ファイル検索：keyword + 10D739C を含むものだけ
    acc_all = glob.glob(os.path.join(acc_path, "*.csv"))
    acc_files = [
        f for f in acc_all
        if (keyword.lower() in os.path.basename(f).lower()) and
           ("10D739C".lower() in os.path.basename(f).lower())
    ]

    if len(acc_files) == 0:
        raise FileNotFoundError(f"keyword='{keyword}', '10D739C' を含む加速度ファイルが見つかりません")

    # 最初の 1 ファイルだけ使用
    acc_file = acc_files[0]
    print(f"Keyword: {keyword} -> Using: {os.path.basename(acc_file)}")

    # --- 加速度ロード ---
    df = pd.read_csv(acc_file)
    df.columns = df.columns.str.strip()

    # 時刻処理
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce").ffill()
    df["time"] = df["ArriveTime"]

    # 数値変換
    df["AccelerationX"] = pd.to_numeric(df.get("AccelerationX"), errors="coerce").fillna(0.0)
    df["AccelerationY"] = pd.to_numeric(df.get("AccelerationY"), errors="coerce").fillna(0.0)
    df["AccelerationZ"] = pd.to_numeric(df.get("AccelerationZ"), errors="coerce").fillna(0.0)

    # --- 特徴量計算 ---
    acc_feat = preprocess_acc_sensor_minimal(
        df, "acc", fs, window_size, bands, center, pad
    )

    # 欠損補完
    if interpolate_missing:
        acc_feat = acc_feat.interpolate("linear").ffill().bfill()

    # datetime 列を除去
    acc_feat = acc_feat.select_dtypes(include=["float", "int"]).ffill()

    # 最初の450データを落とす
    acc_feat = acc_feat.iloc[450:, :].reset_index(drop=True)

    # 標準化
    sc = StandardScaler()
    acc_feat = pd.DataFrame(sc.fit_transform(acc_feat), columns=acc_feat.columns)

    # 保存
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        acc_feat.to_csv(save_path, index=False)
        print("✔ 保存:", save_path)

    return acc_feat


In [ ]:
def generate_log_bands_safe(fs=100, window=128, f_max=25.0, n_bands=12, base=10):
    delta_f = fs / window
    nyquist = fs / 2
    f_max = min(f_max, nyquist)

    # 生のログ境界
    raw_edges = np.logspace(np.log10(delta_f), np.log10(f_max), n_bands + 1, base=base)

    bands = []
    for i in range(n_bands):
        f_lo_raw, f_hi_raw = raw_edges[i], raw_edges[i+1]

        # FFT bin にスナップ
        f_lo_bin = int(np.floor(f_lo_raw / delta_f))
        f_hi_bin = int(np.ceil(f_hi_raw / delta_f))

        # 最低 1 bin を保証
        if f_hi_bin <= f_lo_bin:
            f_hi_bin = f_lo_bin + 1

        f_lo = f_lo_bin * delta_f
        f_hi = f_hi_bin * delta_f

        # Nyquist を越えたら調整
        if f_hi > nyquist:
            f_hi = nyquist
            f_lo = max(delta_f, f_hi - delta_f)

        bands.append((round(f_lo, 4), round(f_hi, 4)))

    return bands

# 描画

In [ ]:
def plot_variables_check(df, columns, title="Plot"):
    for col in columns:
        if col not in df.columns:
            print(f"⚠ {col} は DataFrame にありません")
            continue

        # データ取得
        series = df[col]

        # ビーコン列（カラム名が8桁の16進数）なら 0 補完
        is_beacon = False
        if len(col) == 8:
            try:
                int(col, 16)
                is_beacon = True
            except ValueError:
                pass

        if is_beacon:
            # plot_data = series.fillna(0)
            plot_data = series.interpolate()
        else:
            plot_data = series.interpolate()

        # Figure（1シリーズ = 1枚）
        plt.figure(figsize=(16, 2))
        plt.plot(df.index, plot_data, label=col, linewidth=1.0)

        plt.title(f"{title}: {col}")
        plt.xlabel("time")
        plt.ylabel(col)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# keywords = ['kannno1','kannno2', 'sato1', 'sato2', 'nishi2', 'morita1','morita2','mori1','mori2',"tanaka1","tanaka2","tuji1","tuji2","takagawa1","takagawa2","okawa1","okawa2","maeda1","maeda2"]
keywords = ['kannno1','sato1', 'nishi2','mori1','mori2',"tanaka1","tanaka2","tuji1","tuji2","okawa1","okawa2","maeda1","maeda2"]


windows = [64, 128, 256, 512, 1024, 2048]
# windows = [512,1024]
# windows = [1000]

# 結果格納用
dfs = {}

for wd in windows:
    print(f"------- window_size={wd} ------------")
    bands = generate_log_bands_safe(
        window=wd,
        f_max=25.0,
        n_bands=8   # バンド数を変えると解像度が変わる
    )

    for kw in keywords:
        # 1ファイルだけ処理
        df_key = f"df{kw}_{wd}"
        dfs[df_key] = load_acc_only(
            keyword=kw,
            window_size=wd,
            bands=bands
        )


In [ ]:
# plot_variables(dfs["dfkannno2_128"],columns=dfs["dfmorita1_128"].columns)

In [ ]:
# print(keywords)

# クラスタリング

In [ ]:
num_proc = 2     # 並列化（CPUが12コア）

window_size = 1
number_of_clusters = 2
# beta = 25
beta = 100
maxIters = 30

In [ ]:
def classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc):
    df.to_csv("data/no_heaer.csv", header=False, index=False, na_rep="NaN")
    
    # TICCインスタンスの作成
    ticc = TICC(window_size=window_size,
                number_of_clusters=number_of_clusters,
                beta=beta,
                maxIters=maxIters,
                threshold=2e-5,
                write_out_file=False,
                prefix_string="output_folder/",
                num_proc=num_proc)
    
    # クラスタリングを実行して cluster_assignment を取得
    cluster_assignment, cluster_MRFs = ticc.fit(input_file="data/no_heaer.csv")
    
    # cluster_assignmentの長さがdfと一致しない場合は先頭にNaNで埋める
    if len(cluster_assignment) != len(df):
        pad_len = len(df) - len(cluster_assignment)
        if pad_len > 0:
            # 先頭にnp.nanで埋める（または-1でもOK）
            cluster_assignment_padded = np.concatenate([np.full(pad_len, np.nan), cluster_assignment])
        else:
            raise ValueError(f"cluster_assignment length ({len(cluster_assignment)}) does not match df length ({len(df)})")
    else:
        cluster_assignment_padded = cluster_assignment

    df["class"] = cluster_assignment_padded

    df1 = df
    # df1.to_csv('data/data1_with_class.csv', index=False)
    
    return df1, cluster_assignment_padded

In [ ]:
def plot_variables(df, variables,column = 'class',title=''):
    # color_map = {0: 'blue', 1: 'orange', 2: 'green', 3: 'red', 4: 'purple'}
    color_map = {0: 'blue', 1: 'orange', 4: 'green', 3: 'red', 2: 'purple', 5: 'brown'}

    cluster_series = df[column].values
    change_points = [0] + list(np.where(cluster_series[1:] != cluster_series[:-1])[0] + 1) + [len(df)]

    for var in variables:
        plt.figure(figsize=(20, 2))

        for start, end in zip(change_points[:-1], change_points[1:]):
            cluster = cluster_series[start]

            # NaN のクラスタはスキップ
            if pd.isna(cluster):
                continue

            color = color_map.get(cluster, 'gray')
            plt.plot(df.index[start:end], df[var].iloc[start:end], color=color)

        plt.xlabel('index')
        plt.ylabel(var)
        plt.title(f'{var} with Cluster {title}')
        plt.grid()
        plt.show()

    cluster_series = df['acc_label'].values
    change_points = [0] + list(np.where(cluster_series[1:] != cluster_series[:-1])[0] + 1) + [len(df)]

    for var in variables:
        plt.figure(figsize=(20, 2))

        for start, end in zip(change_points[:-1], change_points[1:]):
            cluster = cluster_series[start]

            # NaN のクラスタはスキップ
            if pd.isna(cluster):
                continue

            color = color_map.get(cluster, 'gray')
            plt.plot(df.index[start:end], df[var].iloc[start:end], color=color)

        plt.xlabel('index')
        plt.ylabel(var)
        plt.title(f'{var} with Label {title}')
        plt.grid()
        plt.show()

In [ ]:
# keywords = ['kannno1','sato1', 'nishi2','morita1','mori1']

for wd in windows:
    for kw in keywords:
        df = dfs[f"df{kw}_{wd}"]
        print(f"------- window_size={wd} ------------")
        print(f"--- keyword={kw} ---")
        # クラスタリング
        classified_df,df_class = classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc)
        df["class"] = df_class
        dfs[f"df{kw}_{wd}"] = df

# 精度評価

In [ ]:
def clustering_accuracy_per_class(df, true_col='acc_label', cluster_col='class'):
    true_labels = df[true_col].to_numpy()
    cluster_labels = df[cluster_col].to_numpy()

    clusters = np.unique(cluster_labels)
    classes = np.unique(true_labels)

    # --- Hungarian のためのコスト行列 ---
    cost_matrix = np.zeros((len(clusters), len(classes)))
    for i, c in enumerate(clusters):
        for j, t in enumerate(classes):
            cost_matrix[i, j] = -np.sum((cluster_labels == c) & (true_labels == t))

    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    # --- クラスタ → 真のラベルの対応 ---
    mapping = {clusters[r]: classes[c] for r, c in zip(row_idx, col_idx)}

    # --- 予測ラベル ---
    predicted = np.array([mapping[c] for c in cluster_labels])

    # --- 全体精度 ---
    overall_acc = accuracy_score(true_labels, predicted)

    # --- クラス別精度 ---
    per_class_acc = {}
    for cls in classes:
        idx = np.where(true_labels == cls)[0]
        per_class_acc[cls] = accuracy_score(true_labels[idx], predicted[idx])
    
    # F1スコアの計算
    f1 = f1_score(true_labels, predicted, average=None)


    return overall_acc, per_class_acc, mapping, predicted, f1

In [ ]:
keywords = ['kannno1','sato1', 'nishi2','mori1','mori2',"tanaka1","tanaka2","tuji1","tuji2","okawa1","okawa2","maeda1","maeda2"]

total_correct = 0
total_samples = 0

major_class_total_correct = 0
major_class_total_samples = 0

minor_class_total_correct = 0
minor_class_total_samples = 0

results = {}

results_all = {}

for k in dfs:
    dfs[k] = dfs[k].bfill()


for wd in windows:
    print(f"   Window Size = {wd} ")
    total_correct = 0
    total_samples = 0
    all_y_true = []
    all_y_pred = []
    for kw in keywords:
        print(f"------- keyword={kw} ------------")
        df = dfs[f"df{kw}_{wd}"]
        
        df_class = df["class"]
        df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")

        df = pd.concat([df_class, df_label], axis=1)
        
        # acc_label列にNaNがある行を削除
        df = df.dropna(subset=['acc_label'])

        # --- 精度計算 ---
        accuracy, per_class_acc, mapping, predicted_labels,f1_scores = clustering_accuracy_per_class(df)

        true_labels = df["acc_label"].to_numpy()

        # --- クラス出現数 ---
        unique, counts = np.unique(true_labels, return_counts=True)

        # 多数派 & 少数派
        major_class = unique[np.argmax(counts)]
        minor_class = unique[np.argmin(counts)]

        # 多数派精度
        major_idx = np.where(true_labels == major_class)[0]
        major_correct = np.sum(predicted_labels[major_idx] == true_labels[major_idx])
        major_acc = major_correct / len(major_idx)

        # 少数派精度
        minor_idx = np.where(true_labels == minor_class)[0]
        minor_correct = np.sum(predicted_labels[minor_idx] == true_labels[minor_idx])
        minor_acc = minor_correct / len(minor_idx)

        # 全体精度
        overall_correct = np.sum(predicted_labels == true_labels)
        overall_acc = overall_correct / len(df)
        # balanced accuracy
        balanced_acc = 0.5 * (major_acc + minor_acc)




        # ===== 人ごとの結果出力 =====
        print(f"Overall accuracy:     {overall_acc:.4f}")
        print(f"Majority class:       {major_class},  accuracy = {major_acc:.4f}")
        print(f"Minority class:       {minor_class},  accuracy = {minor_acc:.4f}")
        print(f"Balanced accuracy:    {balanced_acc:.4f}")
        print(f"F1 Scores:            {f1_scores}")

        # ===== 全体集計に追加 =====
        total_correct += overall_correct
        total_samples += len(df)

        major_class_total_correct += major_correct
        major_class_total_samples += len(major_idx)

        minor_class_total_correct += minor_correct
        minor_class_total_samples += len(minor_idx)

        all_y_true.extend(true_labels)
        all_y_pred.extend(predicted_labels)
        
        results[f"{kw}_{wd}"] = {
            "overall_accuracy": overall_acc,
            "major_class": major_class,
            "major_accuracy": major_acc,
            "minor_class": minor_class,
            "minor_accuracy": minor_acc,
            "balanced_accuracy": balanced_acc,
            "mapping": mapping,
            "f1_scores": f1_scores,
        }

        # plot_variables(df,variables=df.columns[1:2],title=''+kw+'_'+str(wd))

    # ================================
    #     全員分のまとめ
    # ================================
    # overall_accuracy_all = total_correct / total_samples
    # major_accuracy_all = major_class_total_correct / major_class_total_samples
    # minor_accuracy_all = minor_class_total_correct / minor_class_total_samples
    # f1_scores_all = f1_score(all_y_true, all_y_pred, average=None)


    # print("\n===== Accuracy Summary (All People) =====")
    # print(f"Overall accuracy (all samples):      {overall_accuracy_all:.4f}")
    # print(f"Majority-class accuracy (all):       {major_accuracy_all:.4f}")
    # print(f"Minority-class accuracy (all):       {minor_accuracy_all:.4f}")
    # print(f"F1 Scores (all):                     {f1_scores_all}")
    
    # results_all[wd] = {
    #     "overall_accuracy_all": overall_accuracy_all,
    #     "major_accuracy_all": major_accuracy_all,
    #     "minor_accuracy_all": minor_accuracy_all,
    #     "f1_scores_all": f1_scores_all,
    # }

    all_y_true_np = np.asarray(all_y_true, dtype=int)
    all_y_pred_np = np.asarray(all_y_pred, dtype=int)

    overall_accuracy_all = (all_y_true_np == all_y_pred_np).mean()

# 全体で多数/少数を決める
    unique, counts = np.unique(all_y_true_np, return_counts=True)
    major_class = unique[np.argmax(counts)]
    minor_class = unique[np.argmin(counts)]

    major_idx = np.where(all_y_true_np == major_class)[0]
    minor_idx = np.where(all_y_true_np == minor_class)[0]

    major_acc_all = (all_y_pred_np[major_idx] == all_y_true_np[major_idx]).mean()
    minor_acc_all = (all_y_pred_np[minor_idx] == all_y_true_np[minor_idx]).mean()
    balanced_acc_all = 0.5 * (major_acc_all + minor_acc_all)

    # 全体F1スコア
    f1_scores_all = f1_score(all_y_true_np, all_y_pred_np, average="macro",zero_division=0)

    print("\n===== Accuracy Summary (All People) =====")
    print(f"Overall accuracy (all samples):      {overall_accuracy_all:.4f}")
    print(f"Majority-class accuracy (all):       {major_acc_all:.4f}  (class={major_class})")
    print(f"Minority-class accuracy (all):       {minor_acc_all:.4f}  (class={minor_class})")
    print(f"Balanced accuracy (all):             {balanced_acc_all:.4f}")
    print(f"F1 Score (all):                      {f1_scores_all:.4f}")
    print(f"Support: major={len(major_idx)}, minor={len(minor_idx)}")


    results_all[wd] = {
        "overall_accuracy_all": overall_accuracy_all,
        "major_class": major_class,
        "major_accuracy_all": major_acc_all,
        "minor_class": minor_class,
        "minor_accuracy_all": minor_acc_all,
        "balanced_accuracy_all": balanced_acc_all,
        "f1_scores_all": f1_scores_all,
    }

    # 人ごとの結果を棒グラフで比較表示
    labels = []
    overall_accs = []
    major_accs = []
    minor_accs = []
    balanced_accs = []
    avg_f1s = []

    for kw in keywords:
        res = results[f"{kw}_{wd}"]
        labels.append(kw)
        overall_accs.append(res["overall_accuracy"])
        major_accs.append(res["major_accuracy"])
        minor_accs.append(res["minor_accuracy"])
        balanced_accs.append(res["balanced_accuracy"])
        f1_scores = res["f1_scores"]

    x = np.arange(len(labels))  # the label locations
    width = 0.25  # the width of the bars

    fig, ax = plt.subplots(figsize=(12, 6))
    rects1 = ax.bar(x - width, overall_accs, width, label='Overall Accuracy')
    rects2 = ax.bar(x, major_accs, width, label='Majority Class Accuracy')
    rects3 = ax.bar(x + width, minor_accs, width, label='Minority Class Accuracy')
    rects4 = ax.bar(x + 2*width, balanced_accs, width, label='Balanced Accuracy')
    # rects4 = ax.bar(x + 2*width, avg_f1s, width, label='Average F1 Score')

    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Clustering Accuracies by Person (Window Size = {wd})')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.0)
    ax.legend()
    plt.grid(axis='y')
    plt.show()


In [ ]:
# windowごとの精度比較プロット
for kw in keywords:
    major_accs = []
    minor_accs = []
    for wd in windows:
        res = results[f"{kw}_{wd}"]
        major_accs.append(res["major_accuracy"])
        minor_accs.append(res["minor_accuracy"])
    
    plt.figure(figsize=(6, 4))
    plt.plot(windows, major_accs, marker='o', label='Majority Class Accuracy')
    plt.plot(windows, minor_accs, marker='o', label='Minority Class Accuracy')
    plt.xscale('log', base=2)
    plt.xlabel('Window Size')
    plt.ylabel('Accuracy')
    plt.title(f'Accuracy vs Window Size for {kw}')
    plt.xticks(windows, windows)
    plt.ylim(0, 1)
    plt.grid(True, which="both", ls="--", linewidth=0.5)
    plt.legend()
    plt.show()
    
    
# 全体精度比較プロット
major_accs_all = []
minor_accs_all = []
overall_accs_all = []
balanced_acc_all = []  # initialize as list to avoid AttributeError
f1_scores_all = []

for wd in windows:
    res_all = results_all[wd]
    major_accs_all.append(res_all["major_accuracy_all"])
    minor_accs_all.append(res_all["minor_accuracy_all"])
    overall_accs_all.append(res_all["overall_accuracy_all"])
    balanced_acc_all.append(res_all["balanced_accuracy_all"])
    f1_scores_all.append(res_all["f1_scores_all"])
    
plt.figure(figsize=(10, 6))
# plt.plot(windows, overall_accs_all, marker='o', linestyle='-', label='Acc')
# plt.plot(windows, major_accs_all, marker='x', linestyle='--', label='Acc_c0')
# plt.plot(windows, minor_accs_all, marker='s', linestyle='-.', label='Acc_c1')
plt.plot(windows, balanced_acc_all, marker='^', linestyle=':', label='Balanced Accuracy')
plt.plot(windows, f1_scores_all, marker='o', label='F1 Score')
plt.xscale('log', base=2)
plt.xlabel('Window Size',fontsize=15)
plt.ylabel('Accuracy',fontsize=15)
plt.title('Accuracy vs window Size (All People)', fontsize=15)
plt.xticks(windows, windows,fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0.5, 1)
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.legend()
plt.show()

In [ ]:
# windowで人ごとのbalanced Accuracyの精度分布を箱ひげ図でプロット

balanced_accuracies = {wd: [] for wd in windows}
for wd in windows:
    for kw in keywords:
        res = results[f"{kw}_{wd}"]
        major_acc = res["major_accuracy"]
        minor_acc = res["minor_accuracy"]
        balanced_acc = 0.5 * (major_acc + minor_acc)
        balanced_accuracies[wd].append(balanced_acc)
        
plt.figure(figsize=(10, 6))
# 外れ値について考えない箱ひげ図
plt.boxplot(
    [balanced_accuracies[wd] for wd in windows],
    tick_labels=windows,
    whis=(0, 100),   # 最小〜最大までをひげに含める
    showfliers=False # 外れ値マーカーを表示しない
)
plt.xlabel('Window Size',fontsize=15)
plt.ylabel('Balanced Accuracy',fontsize=15)
plt.title('Balanced Accuracies by Window Size', fontsize=15)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, which="both", ls="--", linewidth=0.5)
plt.show()


# バンド幅調整

keywords = ['kannno1','sato1', 'nishi2','morita1','mori1']
# keywords = ['kannno1','kannno2', 'sato1', 'sato2', 'nishi2', 'morita1','morita2','mori1','mori2']
windows = [512]
band_counts = [5,10,15,20]

# 結果格納用
dfb = {}

for wd in windows:
    print(f"------- window_size={wd} ------------")
    for b in band_counts:
        print(f"--- bands={b} ---")
        bands = generate_log_bands_safe(
            window=wd,
            f_max=25.0,
            n_bands=b   # バンド数を変えると解像度が変わる
        )

        for kw in keywords:
            # 1ファイルだけ処理
            df_key = f"df{kw}_{wd}_{b}"
            dfb[df_key] = load_acc_only(
                keyword=kw,
                window_size=wd,
                bands=bands
            )
            
# Fix: iterate over the integer band counts (band_counts) rather than 'bands' (which is a list of tuples),
# and update the correct dfb key. Also add a safety check for missing keys.
for wd in windows:
    for b in band_counts:
        print(f"------- window_size={wd} ------------")
        print(f"--- bands={b} ---")
        for kw in keywords:
            key = f"df{kw}_{wd}_{b}"
            if key not in dfb:
                print(f"⚠ Missing key: {key} -- skipping")
                continue

            df = dfb[key]
            print(f"--- keyword={kw} ---")
            # クラスタリング
            classified_df, df_class = classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc)
            df["class"] = df_class
            dfb[key] = df

total_correct = 0
total_samples = 0

major_class_total_correct = 0
major_class_total_samples = 0

minor_class_total_correct = 0
minor_class_total_samples = 0

results = {}

results_all = {}

for k in dfs:
    dfs[k] = dfs[k].bfill()


for wd in windows:
    print(f"   Window Size = {wd} ")
    # iterate over band_counts (integers used as keys in dfb), not over 'bands' (list of tuples)
    for b in band_counts:
        print(f"--- bands={b} ---")
        for kw in keywords:
            print(f"------- keyword={kw} ------------")
            key = f"df{kw}_{wd}_{b}"
            if key not in dfb:
                print(f"⚠ Missing key: {key} -- skipping")
                continue
            df = dfb[key]
            
            df_class = df["class"]
            df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")

            df = pd.concat([df_class, df_label], axis=1)
            
            # acc_label列にNaNがある行を削除
            df = df.dropna(subset=['acc_label'])

            # --- 精度計算 ---
            accuracy, per_class_acc, mapping, predicted_labels = clustering_accuracy_per_class(df)

            true_labels = df["acc_label"].to_numpy()
            
            # ★ ここで pseudo_class を追加 ★
            df["pseudo_class"] = predicted_labels
            # df and dfb[key] can have different lengths/indexes (labels were merged and rows with NaN dropped).
            # Assign into dfb[key] using the df index so lengths align and avoid ValueError.
            dfb[key].loc[df.index, "pseudo_class"] = predicted_labels
            
            # plot_variables(df,variables=df.columns[1:2],column='pseudo_class',title=''+kw+'_'+str(wd))

            # --- クラス出現数 ---
            unique, counts = np.unique(true_labels, return_counts=True)

            # 多数派 & 少数派
            major_class = unique[np.argmax(counts)]
            minor_class = unique[np.argmin(counts)]

            # 多数派精度
            major_idx = np.where(true_labels == major_class)[0]
            major_correct = np.sum(predicted_labels[major_idx] == true_labels[major_idx])
            major_acc = major_correct / len(major_idx)

            # 少数派精度
            minor_idx = np.where(true_labels == minor_class)[0]
            minor_correct = np.sum(predicted_labels[minor_idx] == true_labels[minor_idx])
            minor_acc = minor_correct / len(minor_idx)

            # 全体精度
            overall_correct = np.sum(predicted_labels == true_labels)
            overall_acc = overall_correct / len(df)

            # ===== 人ごとの結果出力 =====
            print(f"Overall accuracy:     {overall_acc:.4f}")
            print(f"Majority class:       {major_class},  accuracy = {major_acc:.4f}")
            print(f"Minority class:       {minor_class},  accuracy = {minor_acc:.4f}")
            # ===== 全体集計に追加 =====
            total_correct += overall_correct
            total_samples += len(df)
            major_class_total_correct += major_correct
            major_class_total_samples += len(major_idx)
            minor_class_total_correct += minor_correct
            minor_class_total_samples += len(minor_idx)
            results[f"{kw}_{wd}_{b}"] = {
                "overall_accuracy": overall_acc,
                "major_class": major_class,
                "major_accuracy": major_acc,
                "minor_class": minor_class,
                "minor_accuracy": minor_acc,
                "mapping": mapping,
            }


    # ================================
    #     全員分のまとめ
    overall_accuracy_all = total_correct / total_samples
    major_accuracy_all = major_class_total_correct / major_class_total_samples
    minor_accuracy_all = minor_class_total_correct / minor_class_total_samples
    print("\n===== Accuracy Summary (All People) =====")
    print(f"Overall accuracy (all samples):      {overall_accuracy_all:.4f}")
    print(f"Majority-class accuracy (all):       {major_accuracy_all:.4f}")
    print(f"Minor-class accuracy (all):       {minor_accuracy_all:.4f}")


    # store summary keyed by window (and optionally band if desired)
    results_all[f"{wd}"] = {
        "overall_accuracy_all": overall_accuracy_all,
        "major_accuracy_all": major_accuracy_all,
        "minor_accuracy_all": minor_accuracy_all,
    }
    

# bandごとの精度比較プロット
# NOTE: use band_counts (integers) as keys in results (e.g. "kw_wd_20"),
# and guard missing entries to avoid KeyError when 'bands' contains tuples.
for wd in windows:
    for kw in keywords:
        major_accs = []
        minor_accs = []
        for b in band_counts:
            key = f"{kw}_{wd}_{b}"
            if key not in results:
                print(f"⚠ Missing result for {key} -- skipping")
                major_accs.append(np.nan)
                minor_accs.append(np.nan)
                continue
            res = results[key]
            major_accs.append(res["major_accuracy"])
            minor_accs.append(res["minor_accuracy"])
    
        # プロットはバンドを全て集めてから行う（ループ内で描画すると長さ不一致になる）
        plt.figure(figsize=(10, 2))
        plt.plot(band_counts, major_accs, marker='o', label='Majority Class Accuracy')
        plt.plot(band_counts, minor_accs, marker='o', label='Minority Class Accuracy')
        plt.xlabel('Number of Bands')
        plt.ylabel('Accuracy')
        plt.title(f'Accuracy vs Number of Bands for {kw} (Window Size={wd})')
        plt.xticks(band_counts, band_counts)
        plt.ylim(0, 1)
        plt.grid(True, which="both", ls="--", linewidth=0.5)
        plt.legend()
        plt.show()

# 全体精度比較プロット（人物ごとの結果を平均してプロット）
for wd in windows:
    major_accs_all = []
    minor_accs_all = []
    overall_accs_all = []
    for b in band_counts:
        vals_major = []
        vals_minor = []
        vals_overall = []
        for kw in keywords:
            key = f"{kw}_{wd}_{b}"
            if key not in results:
                continue
            r = results[key]
            vals_major.append(r["major_accuracy"])
            vals_minor.append(r["minor_accuracy"])
            vals_overall.append(r["overall_accuracy"])
        if len(vals_major) == 0:
            major_accs_all.append(np.nan)
            minor_accs_all.append(np.nan)
            overall_accs_all.append(np.nan)
        else:
            major_accs_all.append(np.mean(vals_major))
            minor_accs_all.append(np.mean(vals_minor))
            overall_accs_all.append(np.mean(vals_overall))

# 数値表示グラフ
    plt.figure(figsize=(10, 6))
    plt.plot(band_counts, overall_accs_all, marker='o', color = 'red',label='Overall Accuracy')
    plt.plot(band_counts, major_accs_all,color = 'blue', marker='o', label='Majority Class Accuracy')
    plt.plot(band_counts, minor_accs_all,color = 'black' ,marker='o', label='Minority Class Accuracy')

    # --- 数値のラベル ---
    for x, y in zip(band_counts, overall_accs_all):
        plt.text(x, y, f"{y:.2f}", ha='center',color = 'red', va='bottom')

    for x, y in zip(band_counts, major_accs_all):
        plt.text(x, y, f"{y:.2f}", ha='center', color = 'blue',va='top')

    for x, y in zip(band_counts, minor_accs_all):
        plt.text(x, y, f"{y:.2f}", ha='center',color ='black', va='top')

    plt.xlabel('Number of Bands')
    plt.ylabel('Accuracy')
    plt.title(f'Overall Accuracy vs Number of Bands (All People) (Window Size={wd})')
    plt.xticks(band_counts, band_counts)
    plt.ylim(0, 1)
    plt.grid(True, which="both", ls="--", linewidth=0.5)
    plt.legend()
    plt.show()



# 保存

save_root = "../edgeAI実装/class_data_acc"   # ← 後で好きなパスに変更
import os

target_w = "512"
target_b = "10"

# window + bands で 1 フォルダ作成
folder_name = f"window{target_w}_band{target_b}"
folder_path = os.path.join(save_root, folder_name)
os.makedirs(folder_path, exist_ok=True)

for key, df in dfb.items():
    parts = key.split("_")   # ["dfkannno1", "512", "10"]
    if len(parts) != 3:
        continue

    kw_part, wd, b = parts
    if wd == target_w and b == target_b:
        keyword = kw_part.replace("df", "")   # "dfkannno1" → "kannno1"

        save_path = os.path.join(folder_path, f"{keyword}_w{wd}_b{b}.csv")
        df.to_csv(save_path, index=False)
        print("Saved:", save_path)
